# Step 1 · Import a model from Hugging Face

Register an open model as a new base model you can train, evaluate and compare.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
from lab import hub
pd.DataFrame([{"repo": r, "why": why, "cached": hub.is_cached(r)} for r, why in hub.SUGGESTED])

Check size, licence and GPU fit before downloading (needs internet; `hub.info` lifts offline mode just for this call).

In [ ]:
REPO = "Qwen/Qwen2.5-0.5B-Instruct"
try:
    meta = hub.info(REPO); print(meta)
    print(hub.fit_advice(meta["size_gb"]))
except Exception as e:
    print("Offline or not reachable:", type(e).__name__, e)

In [ ]:
# Download once (if needed) and register as a base model
mid = hub.import_model(REPO)
print("registered:", mid)
pd.DataFrame([m for m in registry.models() if m["kind"] == "base"])[["id", "base_model"]]